In [ ]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.dataset_handler import DatasetHandler
from datasets import load_dataset
from utils.data_io import make_dir, save_jsonl
from utils.question_formatter import QuestionFormatter

In [ ]:
dataset = "mmbench"
ds_hdl = DatasetHandler()
ds_hdl.set_curr_ds(dataset)

{'image_scene': {'imgs': 'data/datasets/mmbench/imgs/image_scene',
  'questions': 'data/datasets/mmbench/q_image_scene.jsonl'},
 'image_topic': {'imgs': 'data/datasets/mmbench/imgs/image_topic',
  'questions': 'data/datasets/mmbench/q_image_topic.jsonl'}}

In [3]:
question_formatter = QuestionFormatter(dataset)

In [4]:
ds = load_dataset("lmms-lab/MMBench_EN", split="dev")

In [5]:
ds[0]

{'index': 241,
 'question': "Identify the question that Madelyn and Tucker's experiment can best answer.",
 'hint': 'The passage below describes an experiment. Read the passage and then follow the instructions below.\n\nMadelyn applied a thin layer of wax to the underside of her snowboard and rode the board straight down a hill. Then, she removed the wax and rode the snowboard straight down the hill again. She repeated the rides four more times, alternating whether she rode with a thin layer of wax on the board or not. Her friend Tucker timed each ride. Madelyn and Tucker calculated the average time it took to slide straight down the hill on the snowboard with wax compared to the average time on the snowboard without wax.\nFigure: snowboarding down a hill.',
 'A': "Does Madelyn's snowboard slide down a hill in less time when it has a thin layer of wax or a thick layer of wax?",
 'B': "Does Madelyn's snowboard slide down a hill in less time when it has a layer of wax or when it does not

### Captioning questions

In [6]:
ds_captioning = ds.filter(lambda entry: "caption" in entry["question"])
print(len(ds_captioning))
print(ds_captioning[0])

362
{'index': 34, 'question': 'Which one is the correct caption of this image?', 'hint': 'nan', 'A': 'A woman is riding a motorcycle down the street.', 'B': 'The house appears to be clean and beautifully decorated.', 'C': 'An elephant is chasing a dog around in the dirt.', 'D': 'A woman is walking her dog and has stopped at a corner and is looking at all the lights and buildings.', 'answer': 'A', 'category': 'action_recognition', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x174BCF790>, 'source': 'description', 'l2-category': 'finegrained_perception (cross-instance)', 'comment': 'nan', 'split': 'dev'}


In [7]:
captioning_counts = {}
for entry in ds_captioning:
    if entry["category"] not in captioning_counts:
        captioning_counts[entry["category"]] = 0
    captioning_counts[entry["category"]] += 1

In [8]:
captioning_counts

{'action_recognition': 35,
 'image_scene': 255,
 'object_localization': 8,
 'image_topic': 64}

In [9]:
ds_captioning[0]

{'index': 34,
 'question': 'Which one is the correct caption of this image?',
 'hint': 'nan',
 'A': 'A woman is riding a motorcycle down the street.',
 'B': 'The house appears to be clean and beautifully decorated.',
 'C': 'An elephant is chasing a dog around in the dirt.',
 'D': 'A woman is walking her dog and has stopped at a corner and is looking at all the lights and buildings.',
 'answer': 'A',
 'category': 'action_recognition',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512>,
 'source': 'description',
 'l2-category': 'finegrained_perception (cross-instance)',
 'comment': 'nan',
 'split': 'dev'}

In [ ]:
q = {"image_scene": [], "image_topic": []}

img_path_col = [""] * len(ds_captioning)
ds_captioning = ds_captioning.add_column("img_path", img_path_col)

for q_class in ds_hdl.get_classes():
    ds_hdl.set_curr_class(q_class)
    curr_ds = ds_captioning.filter(lambda entry: entry["category"] == q_class)
    out_dir = Path("..", ds_hdl.get_imgs_path())
    make_dir(out_dir)
    for entry in curr_ds:
        img_path = f"{len(q[q_class])}.jpg"
        entry["img_path"] = str(Path(out_dir, img_path))
        curr_img = entry["image"].convert("RGB")
        curr_img.save(entry["img_path"])
        # Pop image (PIL) before saving question_data to file
        entry.pop("image")
        q[q_class].append(entry)

[Errno 17] File exists: '../data/datasets/mmbench/imgs/image_scene'
[Errno 17] File exists: '../data/datasets/mmbench/imgs/image_topic'


Format and save question data

In [ ]:
for q_class in q:
    ds_hdl.set_curr_class(q_class)
    new_q = [question_formatter.format(raw_q, q_class) for raw_q in q[q_class]]
    save_jsonl(new_q, Path("..", ds_hdl.get_questions_path()))